# 📡 NSE Daily Trade Planner
Tells you **exactly** what to do on the next trading day for every Nifty 100 stock: what to **BUY** (price, quantity, stop-loss), what to **SELL**, and what to **HOLD** (with the updated stop-loss).

**Daily use:** run after **3:35 PM IST** → `Runtime → Run all` (≈2 min). Read section 6.

> ⚠️ Educational tool, not financial advice. Paper trade for 2–3 months first.

## 1. Install

In [ ]:
!pip install yfinance -q

## 2. Your settings
Set `CAPITAL` to the money you'd trade with. Each stock gets one slot = CAPITAL ÷ MAX_POSITIONS, and no trade can lose more than `RISK_PER_TRADE` of CAPITAL if the stop-loss works.

In [ ]:
# ============================ YOUR SETTINGS ============================
UNIVERSE       = "NIFTY100"    # "NIFTY50", "NIFTY100" or "NIFTY200"
HISTORY_START  = "2015-01-01"  # long history = fairer statistics

CAPITAL        = 100_000       # your TOTAL trading money (₹)
MAX_POSITIONS  = 5             # max stocks held at once -> each slot = CAPITAL / MAX_POSITIONS
RISK_PER_TRADE = 0.01          # max loss per trade if stop-loss hits (1% of CAPITAL)

# "Strategy works on this stock?" filter (from its own past backtest)
MIN_SHARPE     = 0.5
MIN_TRADES     = 10

CONFIG = {
    "ema_fast": 20,            # short-term trend line
    "ema_slow": 50,            # medium-term trend line
    "rsi_period": 14,
    "rsi_max_entry": 70,       # skip buys when RSI > 70 (overbought)
    "atr_period": 14,
    "atr_stop_mult": 2.0,      # stop-loss distance = 2 x ATR below price
    "max_gap_atr": 0.5,        # don't chase: skip if next open > close + 0.5 x ATR
    "cost_pct": 0.0015,        # ~0.15% per side (STT 0.1% + stamp + exchange + GST + DP)
}

## 3. Strategy engine
Indicators, backtest (uses the exact same rules as the daily plan), and the plan generator. No need to edit.

In [ ]:
import io, os, datetime as dt
import numpy as np
import pandas as pd

SLOT        = CAPITAL / MAX_POSITIONS         # money per stock
RISK_BUDGET = CAPITAL * RISK_PER_TRADE        # max ₹ loss per trade


# ------------------------------ indicators ------------------------------
def add_indicators(df, c=CONFIG):
    df = df.copy()
    df["ema_fast"] = df["Close"].ewm(span=c["ema_fast"], adjust=False).mean()
    df["ema_slow"] = df["Close"].ewm(span=c["ema_slow"], adjust=False).mean()
    delta = df["Close"].diff()
    gain = delta.clip(lower=0).ewm(alpha=1 / c["rsi_period"], adjust=False).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1 / c["rsi_period"], adjust=False).mean()
    df["rsi"] = 100 - 100 / (1 + gain / loss.replace(0, np.nan))
    pc = df["Close"].shift()
    tr = pd.concat([df["High"] - df["Low"], (df["High"] - pc).abs(), (df["Low"] - pc).abs()], axis=1).max(axis=1)
    df["atr"] = tr.ewm(alpha=1 / c["atr_period"], adjust=False).mean()
    df = df.dropna()
    up = (df["ema_fast"] > df["ema_slow"]) & (df["ema_fast"].shift() <= df["ema_slow"].shift())
    dn = (df["ema_fast"] < df["ema_slow"]) & (df["ema_fast"].shift() >= df["ema_slow"].shift())
    df["buy"] = up & (df["rsi"] < c["rsi_max_entry"])
    df["sell"] = dn
    return df


# ------------------------------ backtest ------------------------------
def backtest(df, c=CONFIG, slot=None, risk_budget=None):
    """
    Exactly the rules the daily plan tells you to follow:
      * signal decided on day t's CLOSE (after 3:30 PM)
      * BUY next morning at the open with a LIMIT = close + 0.5*ATR (skip if it opens higher)
      * stop-loss = entry - 2*ATR, raised every evening to (close - 2*ATR), never lowered
      * exit at stop (or at the open if it gaps below), or at next open after EMA cross-down
    Returns equity curve, trade list and the open position (if any) for tomorrow's plan.
    """
    slot = slot or SLOT
    risk_budget = risk_budget or RISK_BUDGET
    m, cost = c["atr_stop_mult"], c["cost_pct"]
    O, H, L, C = (df[k].values for k in ("Open", "High", "Low", "Close"))
    A, B, S, D = df["atr"].values, df["buy"].values, df["sell"].values, df.index
    cash, qty, entry_px, stop, entry_i = slot, 0, 0.0, 0.0, None
    trades, equity, skipped = [], [], 0

    def close_pos(j, px, why):
        nonlocal cash, qty
        cash += qty * px * (1 - cost)
        pnl = qty * (px * (1 - cost) - entry_px * (1 + cost))
        trades.append({"entry_date": D[entry_i], "entry": entry_px, "exit_date": D[j], "exit": px,
                       "qty": qty, "reason": why, "pnl": pnl,
                       "ret_%": (px * (1 - cost) / (entry_px * (1 + cost)) - 1) * 100,
                       "days": j - entry_i})
        qty = 0

    for i in range(len(df) - 1):
        j = i + 1
        if qty > 0:                                   # 1) stop-loss check
            stop = max(stop, C[i] - m * A[i])
            if L[j] <= stop:
                close_pos(j, min(O[j], stop), "STOP")
        if qty > 0 and S[i]:                          # 2) trend reversal exit
            close_pos(j, O[j], "EMA cross down")
        elif qty == 0 and B[i]:                       # 3) entry
            limit = C[i] + c["max_gap_atr"] * A[i]
            if O[j] > limit:
                skipped += 1
            else:
                risk_ps = m * A[i]
                size = min(int(risk_budget / risk_ps), int(cash / (O[j] * (1 + cost))))
                if size > 0:
                    qty, entry_px, entry_i = size, O[j], j
                    stop = entry_px - risk_ps
                    cash -= qty * entry_px * (1 + cost)
                    if L[j] <= stop:                  # stopped out the same day
                        close_pos(j, stop, "STOP")
        equity.append((D[j], cash + qty * C[j]))

    position = None
    if qty > 0:
        n = len(df) - 1
        position = {"entry_date": D[entry_i], "entry": entry_px, "qty": qty,
                    "stop_tomorrow": max(stop, C[n] - m * A[n]), "days": n - entry_i + 1}
    eq = pd.Series(dict(equity))
    return eq, pd.DataFrame(trades), position, skipped


def stats(eq, tr, df, slot=None):
    slot = slot or SLOT
    years = max((eq.index[-1] - eq.index[0]).days / 365.25, 0.1)
    r = eq.pct_change().dropna()
    bh = df["Close"].iloc[-1] / df["Close"].loc[eq.index[0]:].iloc[0]
    n = len(tr)
    in_mkt = tr["days"].sum() / max(len(eq), 1) * 100 if n else 0.0
    return {
        "Strategy CAGR %": ((eq.iloc[-1] / slot) ** (1 / years) - 1) * 100,
        "Buy&Hold CAGR %": (bh ** (1 / years) - 1) * 100,
        "Sharpe": np.sqrt(252) * r.mean() / r.std() if r.std() > 0 else 0.0,
        "MaxDD %": (eq / eq.cummax() - 1).min() * 100,
        "Trades": n,
        "Win %": (tr["pnl"] > 0).mean() * 100 if n else 0.0,
        "Avg win %": tr.loc[tr["pnl"] > 0, "ret_%"].mean() if n and (tr["pnl"] > 0).any() else 0.0,
        "Avg loss %": tr.loc[tr["pnl"] <= 0, "ret_%"].mean() if n and (tr["pnl"] <= 0).any() else 0.0,
        "Avg hold days": tr["days"].mean() if n else 0.0,
        "Time in market %": in_mkt,
    }


# ------------------------------ tomorrow's plan ------------------------------
def next_trading_day(d):
    return (pd.Timestamp(d) + pd.offsets.BDay(1)).date()


def trade_plan(sym, df, tr, position, c=CONFIG):
    """Exact instructions for the next trading day."""
    last, day = df.iloc[-1], df.index[-1]
    nxt = next_trading_day(day).strftime("%a %d-%b")
    m = c["atr_stop_mult"]
    close, atr = last["Close"], last["atr"]
    gap = (last["ema_fast"] / last["ema_slow"] - 1) * 100
    p = {"Symbol": sym, "Close": round(close, 2), "RSI": round(last["rsi"], 1),
         "EMA gap %": round(gap, 2), "Next day": nxt,
         "Buy limit": None, "Qty": None, "Stop-loss": None, "Risk ₹": None,
         "Entry": None, "Entry date": None, "Day #": None, "P/L %": None, "Plan": ""}

    exited_today = len(tr) and tr.iloc[-1]["exit_date"] == day
    if position:
        e = position
        pl = (close / e["entry"] - 1) * 100
        p.update({"Entry": round(e["entry"], 2), "Entry date": e["entry_date"].date(),
                  "Day #": e["days"], "P/L %": round(pl, 2),
                  "Stop-loss": round(e["stop_tomorrow"], 2), "Qty": e["qty"]})
        if last["sell"]:
            p["Action"] = "🔴 SELL"
            p["Plan"] = (f"Trend reversed today (EMA{c['ema_fast']} crossed below EMA{c['ema_slow']}). "
                         f"SELL all {e['qty']} shares on {nxt} at the open (9:15-9:20 AM, market/limit at open) "
                         f"and cancel your GTT stop-loss. Trade result so far: {pl:+.1f}% in {e['days']} days.")
        else:
            p["Action"] = "🟡 HOLD"
            dist = (close / e["stop_tomorrow"] - 1) * 100
            p["Plan"] = (f"Day {e['days']} of the trade (bought {e['entry_date'].date()} @ ₹{e['entry']:.2f}, now {pl:+.1f}%). "
                         f"Do nothing except: update your GTT stop-loss to ₹{e['stop_tomorrow']:.2f} "
                         f"({dist:.1f}% below today's close). Exit early only if EMA{c['ema_fast']} closes below EMA{c['ema_slow']} "
                         f"(gap now {gap:+.2f}%). Didn't buy on the signal day? Don't buy now; wait for a fresh BUY.")
    elif exited_today:
        t = tr.iloc[-1]
        p["Action"] = "🛑 STOPPED" if t["reason"] == "STOP" else "⚪ EXITED"
        p["Plan"] = (f"Strategy exited today at ₹{t['exit']:.2f} ({t['reason']}, {t['ret_%']:+.1f}%). "
                     f"If your GTT didn't execute and you still hold it, SELL at the open on {nxt}.")
    elif last["buy"]:
        risk_ps = m * atr
        limit = close + c["max_gap_atr"] * atr
        qty = min(int(RISK_BUDGET / risk_ps), int(SLOT / limit))
        stop = close - risk_ps
        p.update({"Action": "🟢 BUY", "Buy limit": round(limit, 2), "Qty": qty,
                  "Stop-loss": round(stop, 2), "Risk ₹": round(qty * risk_ps)})
        if qty == 0:
            p["Plan"] = (f"BUY signal, but 1 share (₹{close:,.0f}) is more than your slot (₹{SLOT:,.0f}) "
                         f"or risk budget allows. Skip, or increase CAPITAL.")
        else:
            p["Plan"] = (f"On {nxt}, 9:15-9:30 AM: place a LIMIT BUY (CNC/Delivery) for {qty} shares at up to ₹{limit:.2f}. "
                         f"If it opens above ₹{limit:.2f}, SKIP (don't chase). "
                         f"Right after it fills, place a GTT stop-loss SELL at ₹{stop:.2f} "
                         f"(= your buy price minus ₹{risk_ps:.2f}). Max loss ≈ ₹{qty * risk_ps:,.0f} "
                         f"({qty * risk_ps / CAPITAL * 100:.1f}% of capital). Cost ≈ ₹{qty * close:,.0f}.")
    else:
        if gap < 0 and gap > -1.0 and last["ema_fast"] > df["ema_fast"].iloc[-2]:
            p["Action"] = "👀 NEAR BUY"
            p["Plan"] = (f"EMA{c['ema_fast']} is only {abs(gap):.2f}% below EMA{c['ema_slow']} and rising. "
                         f"A BUY signal may come in the next few days. Watch only, don't buy yet.")
        elif gap > 0:
            p["Action"] = "⚪ WAIT"
            p["Plan"] = "Uptrend already running but no fresh crossover. Entering late = worse risk/reward. Wait."
        else:
            p["Action"] = "⚪ WAIT"
            p["Plan"] = "Downtrend (EMA fast below slow). Stay away until a BUY signal."
    return p


# ------------------------------ universe & data ------------------------------
FALLBACK = """ADANIENT ADANIPORTS APOLLOHOSP ASIANPAINT AXISBANK BAJAJ-AUTO BAJFINANCE BAJAJFINSV BEL BHARTIARTL
CIPLA COALINDIA DRREDDY EICHERMOT ETERNAL GRASIM HCLTECH HDFCBANK HDFCLIFE HINDALCO HINDUNILVR ICICIBANK
INDIGO INFY ITC JIOFIN JSWSTEEL KOTAKBANK LT M&M MARUTI MAXHEALTH NESTLEIND NTPC ONGC POWERGRID RELIANCE
SBILIFE SBIN SHRIRAMFIN SUNPHARMA TATACONSUM TATASTEEL TCS TECHM TITAN TRENT ULTRACEMCO WIPRO
DMART HAL DIVISLAB PIDILITIND SIEMENS HAVELLS DABUR GODREJCP BRITANNIA LTIM VEDL TVSMOTOR BANKBARODA PNB
CANBK IOC BPCL GAIL TATAPOWER ADANIGREEN ADANIPOWER AMBUJACEM SHREECEM DLF LODHA ICICIGI ICICIPRULI
CHOLAFIN BAJAJHLDNG PFC RECLTD IRFC ZYDUSLIFE TORNTPHARM MOTHERSON BOSCHLTD NAUKRI INDHOTEL UNITDSPR
VBL JSWENERGY ABB CGPOWER LICI HINDZINC BSE""".split()


def load_universe(name=UNIVERSE):
    import requests
    f = f"ind_{name.lower()}list.csv"
    for url in (f"https://niftyindices.com/IndexConstituent/{f}",
                f"https://archives.nseindia.com/content/indices/{f}"):
        try:
            r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
            r.raise_for_status()
            u = pd.read_csv(io.StringIO(r.text)).rename(columns={"Company Name": "Company"})
            print(f"✅ Loaded {len(u)} stocks in {name} from NSE")
            return u[["Symbol", "Company", "Industry"]]
        except Exception:
            continue
    print(f"⚠️ Couldn't reach NSE, using built-in list of {len(FALLBACK)} large caps")
    return pd.DataFrame({"Symbol": FALLBACK, "Company": FALLBACK, "Industry": ""})


def completed_candles(d):
    """Drop today's candle if the market hasn't closed yet (avoids fake signals)."""
    now = pd.Timestamp.now(tz="Asia/Kolkata")
    if len(d) and d.index[-1].date() == now.date() and now.time() < dt.time(15, 35):
        return d.iloc[:-1]
    return d


def download_all(symbols, start=HISTORY_START, batch=50):
    import yfinance as yf
    data, tickers = {}, [s + ".NS" for s in symbols]
    for k in range(0, len(tickers), batch):
        chunk = tickers[k:k + batch]
        raw = yf.download(chunk, start=start, group_by="ticker", auto_adjust=True,
                          threads=True, progress=False)
        for t in chunk:
            try:
                d = completed_candles(raw[t][["Open", "High", "Low", "Close", "Volume"]].dropna())
                if len(d) > 200:
                    data[t[:-3]] = d
            except Exception:
                pass
        print(f"Downloaded {min(k + batch, len(tickers))}/{len(tickers)}")
    missing = sorted(set(symbols) - set(data))
    if missing:
        print("⚠️ No data for:", ", ".join(missing))
    return data


def fundamentals(sym):
    import yfinance as yf
    try:
        i = yf.Ticker(sym + ".NS").info
        pm, roe = i.get("profitMargins"), i.get("returnOnEquity")
        return {"Profit margin %": round(pm * 100, 1) if pm is not None else None,
                "ROE %": round(roe * 100, 1) if roe is not None else None,
                "Profitable?": "✅" if (pm or 0) > 0 and (roe or 0) > 0.12 else "❌"}
    except Exception:
        return {"Profit margin %": None, "ROE %": None, "Profitable?": "?"}


# ------------------------------ full scan ------------------------------
ORDER = {"🟢 BUY": 0, "🔴 SELL": 1, "🛑 STOPPED": 2, "🟡 HOLD": 3, "👀 NEAR BUY": 4, "⚪ EXITED": 5, "⚪ WAIT": 6}


def run_scan(universe, prices):
    rows, latest = [], max(d.index[-1] for d in prices.values())
    for sym, raw in prices.items():
        try:
            df = add_indicators(raw)
            eq, tr, pos, _ = backtest(df)
            st = stats(eq, tr, df)
            plan = trade_plan(sym, df, tr, pos)
            works = st["Sharpe"] >= MIN_SHARPE and st["Trades"] >= MIN_TRADES and st["Strategy CAGR %"] > 0
            rows.append({**plan, "Strategy works?": "✅" if works else "❌",
                         **{k: round(v, 2) for k, v in st.items()},
                         "Data date": df.index[-1].date(), "Stale": df.index[-1] < latest})
        except Exception as e:
            print(sym, "error:", e)
    scan = (pd.DataFrame(rows).merge(universe[["Symbol", "Company", "Industry"]], on="Symbol", how="left")
            .set_index("Symbol"))
    scan["_o"] = scan["Action"].map(ORDER)
    scan = scan.sort_values(["_o", "Sharpe"], ascending=[True, False]).drop(columns="_o")
    for s in scan.index[scan["Action"] == "🟢 BUY"]:
        f = fundamentals(s)
        for k, v in f.items():
            scan.loc[s, k] = v
    if "Profitable?" in scan:
        scan["⭐"] = ((scan["Action"] == "🟢 BUY") & (scan["Strategy works?"] == "✅")
                     & (scan["Profitable?"] == "✅")).map({True: "⭐", False: ""})
    else:
        scan["⭐"] = ""
    return scan, latest


def save_results(scan, latest, out="scanner_output"):
    os.makedirs(out, exist_ok=True)
    day = latest.date().isoformat()
    scan.to_csv(f"{out}/scan_latest.csv")
    log_path = f"{out}/signal_log.csv"
    cols = ["Action", "Company", "Close", "Buy limit", "Qty", "Stop-loss", "Entry", "P/L %", "Strategy works?"]
    new = scan[scan["Action"].isin(["🟢 BUY", "🔴 SELL", "🛑 STOPPED"])].reset_index()[["Symbol"] + cols]
    new.insert(0, "Signal date", day)
    if os.path.exists(log_path):
        old = pd.read_csv(log_path)
        new = pd.concat([old[old["Signal date"] != day], new], ignore_index=True)
    new.to_csv(log_path, index=False)
    return f"{out}/scan_latest.csv"

## 4. Scan all stocks

In [ ]:
universe = load_universe(UNIVERSE)
prices = download_all(universe["Symbol"].tolist())
scan, latest = run_scan(universe, prices)
print(f"\nPlan for {next_trading_day(latest):%a %d-%b}  (based on {latest.date()} close)")
print(scan["Action"].value_counts().to_string())
stale = scan.index[scan["Stale"]].tolist()
if stale:
    print("⚠️ Older data (check manually):", ", ".join(stale))

## 5. Today's actions — your plan for tomorrow

In [ ]:
from IPython.display import display, Markdown


def show_section(action, title):
    part = scan[scan["Action"] == action]
    display(Markdown(f"### {title} ({len(part)})"))
    if not len(part):
        print("None today."); return
    for s, r in part.iterrows():
        star = r.get("⭐", "")
        hist = (f"history on this stock: CAGR {r['Strategy CAGR %']:.1f}%/yr, win {r['Win %']:.0f}%, "
                f"avg hold {r['Avg hold days']:.0f} days, strategy works? {r['Strategy works?']}")
        display(Markdown(f"**{star} {s}** – {r['Company']}  \n{r['Plan']}  \n<sub>{hist}</sub>"))


show_section("🟢 BUY", "🟢 BUY tomorrow")
show_section("🔴 SELL", "🔴 SELL tomorrow at the open")
show_section("🛑 STOPPED", "🛑 Stop-loss hit today")
show_section("🟡 HOLD", "🟡 HOLD (only if you own it)")
show_section("👀 NEAR BUY", "👀 Watchlist")

## 6. Deep dive into one stock
Chart with past buys (▲) / sells (▼), stop-loss line and RSI, plus its last 10 trades.

In [ ]:
import matplotlib.pyplot as plt


def deep_dive(sym, months=12):
    """Chart + every past trade for one stock, so you can SEE the rules working."""
    df = add_indicators(prices[sym])
    eq, tr, pos, skipped = backtest(df)
    plan = trade_plan(sym, df, tr, pos)
    d = df.iloc[-months * 21:]
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
    a1.plot(d.index, d["Close"], color="#333", lw=1.2, label="Close")
    a1.plot(d.index, d["ema_fast"], color="#1f77b4", lw=1, label=f"EMA{CONFIG['ema_fast']}")
    a1.plot(d.index, d["ema_slow"], color="#ff7f0e", lw=1, label=f"EMA{CONFIG['ema_slow']}")
    t = tr[tr["entry_date"] >= d.index[0]] if len(tr) else tr
    if len(t):
        a1.scatter(t["entry_date"], t["entry"], marker="^", s=90, c="green", zorder=5, label="Buy")
        a1.scatter(t["exit_date"], t["exit"], marker="v", s=90, c="red", zorder=5, label="Sell/Stop")
    if pos:
        a1.axhline(pos["stop_tomorrow"], color="red", ls="--", lw=1, label=f"Stop ₹{pos['stop_tomorrow']:.0f}")
    a1.set_title(f"{sym}: {plan['Action'].split(' ', 1)[1]}"); a1.legend(loc="upper left"); a1.grid(alpha=.3)
    a2.plot(d.index, d["rsi"], color="purple"); a2.axhline(70, color="red", ls=":"); a2.axhline(30, color="green", ls=":")
    a2.set_ylabel("RSI"); a2.grid(alpha=.3)
    plt.tight_layout(); plt.show()
    print("PLAN:", plan["Plan"])
    st = stats(eq, tr, df)
    print({k: round(v, 2) for k, v in st.items()}, f"| entries skipped (gap-up): {skipped}")
    return tr.tail(10)


deep_dive(scan.index[0])        # change to any symbol, e.g. deep_dive("TCS")

## 7. Save results + paper-trading log

In [ ]:
SAVE_TO_DRIVE = False           # True = keep a daily history in Google Drive
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/algo_scanner"
else:
    OUT = "scanner_output"
print("Saved:", save_results(scan, latest, OUT), "+ signal_log.csv")

## 8. Full table

In [ ]:
scan